# Explainable AI with SHAP (SHapley Additive exPlanations)

**A beginner-friendly, hands-on Colab notebook**

You do NOT need any prior experience with explainable AI to follow this notebook. Every chart
comes with a plain-English, step-by-step "how to read this" walkthrough that points at the
*exact* colors, numbers, and axes you'll see on screen.

**Roadmap:**
1. Why explainability matters
2. Train a simple, realistic classifier (customer churn prediction)
3. Compute SHAP values
4. **Global explanations** — what the model cares about overall (2 chart types)
5. **Local explanations** — why the model made *this* prediction for *this* customer (2 chart types)
6. **Interaction analysis** — does a feature behave differently depending on another feature?
7. A quick look at SHAP for a non-tree model
8. Summary + discussion questions

> **Estimated runtime:** ~2–3 minutes end-to-end on Colab CPU. No GPU, no file uploads needed.


## 1. Setup

In [ ]:
# Install SHAP (usually not preinstalled on Colab)
!pip install shap --quiet


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import shap
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.linear_model import LogisticRegression

shap.initjs()  # enables interactive JS plots in classic Jupyter (see note in Section 9)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("SHAP version:", shap.__version__)


## 2. The dataset: synthetic customer churn

**What is "churn"?** In business, *churn* means a customer stops using a service (e.g. cancels
their subscription). Predicting churn ahead of time lets a company reach out and try to retain
the customer.

We simulate this problem with 8 features that sound like real business metrics. Importantly,
we **deliberately build in realistic cause-and-effect relationships** (e.g. "lower satisfaction
→ higher churn chance") using a simple weighted formula, then add randomness on top. This
matters for learning: it means the SHAP charts later on will show patterns that make real
business sense, instead of arbitrary noise — so when you read "high monthly charges pushes
this customer toward churn," that's a genuine, traceable pattern in the data, not a
coincidence.

**The 8 features we'll use, and the effect we've built into each one:**

| Feature | Meaning | Effect on churn we've built in |
|---|---|---|
| `tenure_months` | Months as a customer | Longer tenure → **lower** churn chance |
| `monthly_charges` | Amount billed per month | Higher charges → **higher** churn chance |
| `total_charges` | Total billed so far (≈ tenure × monthly charges) | No independent effect — included to show a *derived/redundant* feature |
| `num_support_tickets` | Number of support contacts | More tickets → **higher** churn chance |
| `contract_length_months` | Contract type: 1, 12, or 24 months | Longer contract → **lower** churn chance |
| `satisfaction_score` | Self-reported score, 0 (low) to 10 (high) | Higher satisfaction → **lower** churn chance |
| `num_products` | Number of products/services subscribed | More products → **lower** churn chance (more "stickiness") |
| `late_payments_last_year` | Late payments in the past year | More late payments → **higher** churn chance |

**The target:** `churned` — 1 if the customer left, 0 if they stayed. Keep this table close by
— every chart from here on is really just a different way of *proving* (or measuring) these
relationships from the model's point of view.


In [ ]:
n = 2000

tenure_months = np.random.uniform(0, 72, n)
monthly_charges = np.random.uniform(20, 150, n)
num_support_tickets = np.random.poisson(2, n).clip(0, 15)
contract_length_months = np.random.choice([1, 12, 24], size=n, p=[0.4, 0.35, 0.25])
satisfaction_score = np.random.uniform(0, 10, n)
num_products = np.random.randint(1, 6, n)
late_payments_last_year = np.random.poisson(1, n).clip(0, 10)
total_charges = (tenure_months * monthly_charges * (0.9 + 0.2 * np.random.rand(n))).round(2)

def zscore(x):
    x = np.asarray(x, dtype=float)
    return (x - x.mean()) / x.std()

# Combine features into a single "churn score" (logit) with hand-picked weights,
# matching the effects described in the table above. This is what makes the
# dataset's patterns realistic and consistent, rather than arbitrary.
logit = (
    -1.1 * zscore(satisfaction_score)
    + 0.9 * zscore(monthly_charges)
    + 0.6 * zscore(num_support_tickets)
    - 0.7 * zscore(contract_length_months)
    - 0.5 * zscore(tenure_months)
    - 0.4 * zscore(num_products)
    + 0.6 * zscore(late_payments_last_year)
    - 1.4  # intercept: controls the overall (base) churn rate
)
churn_probability = 1 / (1 + np.exp(-logit))
churned = np.random.binomial(1, churn_probability)

X = pd.DataFrame({
    "tenure_months": tenure_months.round(0),
    "monthly_charges": monthly_charges.round(2),
    "total_charges": total_charges,
    "num_support_tickets": num_support_tickets,
    "contract_length_months": contract_length_months,
    "satisfaction_score": satisfaction_score.round(1),
    "num_products": num_products,
    "late_payments_last_year": late_payments_last_year,
})
y = pd.Series(churned, name="churned")

print("Rows:", X.shape[0], " | Overall churn rate:", round(y.mean(), 3))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

X.head()


**📖 What am I looking at?** The table above (`X.head()`) shows the first 5 customers, one
row each, one column per feature — just the raw input data. Nothing to interpret yet; this is
just to get familiar with the columns before we train a model on them.


## 3. Train the model

We use a `RandomForestClassifier` — a model made of many decision trees that vote together. We
picked a tree-based model on purpose: SHAP has an extremely fast, exact explainer
(`TreeExplainer`) for tree models, which makes it the easiest starting point for learning SHAP.

Importantly: **we only give the model the raw features (`X_train`) and the outcome
(`y_train`)** — it never sees our hand-picked weights from Section 2. Everything the model
"learns" about which features matter, it has to discover purely from the data. That's what
makes the SHAP charts later genuinely interesting: we get to check whether the model correctly
rediscovered the relationships we built in.


In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=20,
    random_state=RANDOM_STATE,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["stayed", "churned"]))
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 3))


**📖 What am I looking at?** This is a standard performance report, not a SHAP output yet —
but it's important context for everything that follows.

- **Precision / recall / f1-score**: how good the model is at correctly catching churners vs.
  raising false alarms. Higher is better; 1.0 would be a perfect (and suspicious!) model.
- **ROC-AUC** (typically ~0.85–0.90 here): a single number from 0.5 (random guessing) to 1.0
  (perfect separation). Our model is doing a solid job separating churners from non-churners.

**Why show this before explainability?** Because explainability doesn't replace measuring
accuracy — it *complements* it. A model can score well here and still be a black box about
*why* it makes each decision. That's the problem SHAP solves next.


## 4. Why isn't accuracy/AUC enough?

A model can be accurate and still be a black box. Stakeholders — a bank regulator, a customer,
a course assessor — often need answers to two different kinds of questions:

- **Global question:** *"Which features drive predictions overall?"* → helps you check the model
  isn't relying on something silly, unfair, or biased.
- **Local question:** *"Why did the model flag THIS specific customer as high-risk?"* → needed
  for customer service scripts, appeals, and building trust with the people affected by the
  decision.

**The key idea behind SHAP:** it's based on Shapley values, a concept from game theory
originally designed to fairly split a prize among players who worked together. SHAP treats each
**feature as a "player"** and calculates each player's **fair share of credit (or blame)** for
the model's prediction, compared to the model's average prediction across all customers.

You don't need to know the underlying math to use SHAP — just remember this one sentence,
because every chart in this notebook is a different way of visualizing it:

> **A SHAP value tells you how much one feature pushed one prediction up or down, compared to
> the average prediction.**

One more important distinction before we start:

- **A feature's SHAP value** is about its *influence on the prediction* (can be positive,
  negative, or near zero).
- **A feature's actual value** (e.g. `satisfaction_score = 2.3`) is just the raw data for that
  customer.

Every chart below shows *both* of these — and beginners most often get confused mixing the two
up. We'll flag clearly, every time, which is which.


## 5. Computing SHAP values with `TreeExplainer`

`TreeExplainer` is exact and fast for tree ensembles (Random Forest, XGBoost, LightGBM,
CatBoost). Run the cell below to calculate a SHAP value for **every feature, for every
customer** in our test set. Think of it as a giant spreadsheet: one row per customer, one
column per feature, and each cell answers "how much did this feature push this customer's
prediction up or down?"


In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test)   # new-style Explanation object (SHAP >= 0.40)

# shap_values has shape (n_samples, n_features, n_classes) for multi-output models.
# We'll focus on the "churned" class (class index 1).
shap_values_churn = shap_values[..., 1]

print("Explanation object shape:", shap_values.values.shape)


**📖 What am I looking at?** The printed shape, e.g. `(500, 8, 2)`, means: 500 test
customers × 8 features × 2 classes (stayed / churned). We narrowed it down to
`shap_values_churn`, which just keeps the "churned" numbers — that's the one we'll use for
every chart below. No chart yet; this cell just runs the calculation that all the visuals will
be built from.


## 6. Global explanation #1 — Feature importance (bar chart)

**Question this chart answers:** *"Across ALL customers, which features matter most to the
model, on average?"*

This ranks features by their **average impact on model output magnitude** — a more faithful
alternative to `feature_importances_`, because it accounts for actual contribution to
predictions, not just how often a feature was used to split a tree.


In [ ]:
shap.summary_plot(shap_values_churn, X_test, plot_type="bar", show=False)
plt.title("Global feature importance (mean |SHAP value|)")
plt.tight_layout()
plt.show()


**📖 How to read this chart, step by step:**

1. **Y-axis (left side, the labels):** each row is one feature — e.g. `monthly_charges`,
   `satisfaction_score`.
2. **X-axis (bottom):** labeled `mean(|SHAP value|)`. This is the *average size* of that
   feature's SHAP value across every customer in the test set, ignoring direction (that's what
   the `|...|`, "absolute value," means). A bigger bar = bigger average influence on the
   prediction, whether that influence pushes toward or away from churn.
3. **Order matters:** features are sorted top-to-bottom, most to least important. Whatever is
   at the top is the single biggest driver of churn predictions overall, according to the
   model.
4. **Sanity-check against Section 2's table:** you should see `monthly_charges`,
   `satisfaction_score`, `num_support_tickets`, `contract_length_months`, `tenure_months`,
   `num_products`, and `late_payments_last_year` all appear — because we built a real effect
   for each of them into the data. `total_charges` may still show up because it's correlated
   with `tenure_months` and `monthly_charges`, even without an independent effect of its own —
   a nice preview of why correlated features can be tricky to interpret.
5. **What this chart does NOT tell you:** it doesn't say whether a feature pushes predictions
   *toward* churn or *away from* churn — only how much it moves the needle either way. That
   direction is exactly what the next chart (the beeswarm plot) adds.

**Takeaway:** use this chart as your "top-line summary" — a quick answer to "what does the
model care about?" before diving into more detailed charts.


## 7. Global explanation #2 — Beeswarm plot

**Question this chart answers:** *"For the top features, HOW do they push predictions — and
for which customers?"* This uses the same ranking as the bar chart, but layers on two more
pieces of information: direction, and each customer's actual value.


In [ ]:
shap.summary_plot(shap_values_churn, X_test, show=False)
plt.title("SHAP summary (beeswarm) — churn class")
plt.tight_layout()
plt.show()


**📖 How to read this chart, step by step:**

1. **Rows = features**, same top-to-bottom ranking as the bar chart (most important at top).
2. **Each dot = one customer.** With 500 test customers, you'll see roughly 500 dots per row,
   clustered together like a swarm of bees (hence the name).
3. **Horizontal position (x-axis) = that customer's SHAP value for that feature.**
   - A dot to the **right of the vertical zero line** means: for this customer, this feature
     pushed the churn prediction **up** (toward churn).
   - A dot to the **left of zero** means this feature pushed the prediction **down** (toward
     "stayed").
   - The further from zero, the stronger the push — near zero means "barely mattered for this
     customer."
4. **Color = the feature's actual value for that customer** (this is the "actual data," not
   the SHAP value):
   - 🔴 **Pink/red = high value** of that feature.
   - 🔵 **Blue = low value** of that feature.
5. **Put it together, one row at a time.** Look at the `satisfaction_score` row: you should see
   mostly **blue dots (low satisfaction) sitting to the right** (pushing toward churn) and
   mostly **red/pink dots (high satisfaction) sitting to the left** (pushing away from churn).
   That confirms the relationship we built into the data in Section 2: *low satisfaction
   increases predicted churn risk* — and now we can see the model correctly learned it, purely
   from examples, without ever being told the rule directly.
6. **Try it yourself on another row**, e.g. `monthly_charges`: you should find red/pink
   (high charges) dots mostly to the right (toward churn) and blue (low charges) dots mostly to
   the left (toward staying) — matching "higher charges → higher churn chance" from our table.
7. **Not every row will look like a perfectly clean split.** A more mixed cloud (e.g. for
   `total_charges`) usually means that feature's effect is tangled up with another correlated
   feature — we investigate that kind of pattern in Section 10.

**Takeaway:** the beeswarm plot is the single most information-dense SHAP chart. If you can
only show learners one plot, this is often the one.


## 8. Local explanation — Waterfall plot for a single customer

So far every chart summarized the *whole* test set. Now we zoom into **one single customer**
and ask: *"Why did the model give THIS customer THIS specific prediction?"*

This is the plot you'd show a customer-service rep: *"here's exactly why the model thinks this
customer might churn."*


In [ ]:
# Pick the test customer with the highest predicted churn probability
idx = int(np.argmax(y_proba))
print(f"Explaining test row {idx} — predicted churn probability: {y_proba[idx]:.2%}")

shap.plots.waterfall(shap_values_churn[idx], show=False)
plt.tight_layout()
plt.show()


**📖 How to read this chart, step by step — this one has the most moving parts, so let's
be extra precise:**

There are actually **three different kinds of numbers** on this chart. Beginners often
conflate them, so look for each one specifically:

1. **The gray numbers on the far left** (e.g. `0.3 = satisfaction_score`) are this customer's
   **actual data values** — exactly what's stored in the spreadsheet for this one customer.
   This is NOT a SHAP value; it's just the raw input. (This customer's satisfaction score is a
   very low 0.3 out of 10.)
2. **The bold number printed inside or beside each colored bar** (e.g. `+0.16` or `+0.12`) IS
   the **SHAP value** — how many percentage points of predicted churn probability that feature
   added or removed for this customer. For example, `+0.16` next to `satisfaction_score` means
   "this customer's very low satisfaction score alone added 16 percentage points to their churn
   probability."
3. **The bottom axis (`E[f(X)] = 0.293`)** is the model's *average* predicted churn probability
   across **all** customers — the neutral starting point before we know anything about this
   specific person. (On average, the model expects a 29.3% chance of churn for any random
   customer.)

**Now read the chart itself, top to bottom in the order the bars are drawn:**

4. **Start at the bottom of the staircase**, at `E[f(X)] = 0.293` — think of this as "before we
   knew anything about this customer, the model's best guess was a 29.3% churn chance."
5. **Each bar adds one feature's effect**, moving the running total left or right:
   - 🔴 **Red bars pointing right** = this feature's value pushed the prediction **up**
     (increased churn probability) for this customer. In this chart, every bar happens to be
     red — every one of this customer's feature values pushed them further toward churn, which
     is exactly why their final risk is so high.
   - 🔵 **Blue bars pointing left** = this feature's value pushed the prediction **down**
     (decreased churn probability). You'll see blue bars for other, lower-risk customers — try
     changing `idx` in the code above to explore one.
   - The **width of the bar** matches the size of its printed SHAP value — wider bar = bigger
     effect. Notice `satisfaction_score`'s bar (`+0.16`) is visibly wider than `num_products`'s
     bar (`+0.02`).
6. **Bars are stacked in order of impact**, largest at the top (closest to the final value),
   smallest near the bottom — so you can scan top-to-bottom for "the biggest reasons first."
   Here that order is: `satisfaction_score` (+0.16) → `monthly_charges` (+0.12) →
   `tenure_months` (+0.07) → `late_payments_last_year` (+0.05) → `contract_length_months`
   (+0.05) → `num_support_tickets` (+0.04) → `num_products` (+0.02) → `total_charges` (+0.00,
   essentially no effect once its correlated cousins `tenure_months` and `monthly_charges` are
   already accounted for).
7. **The top of the staircase, `f(x) = 0.799`**, is the final prediction for this one customer
   — a 79.9% predicted churn probability. It should match (or be very close to) the
   `predicted churn probability` printed above the chart in the code output (here: 79.93%).
8. **Connect the dots between the three number types:** e.g. `135.64 = monthly_charges` on the
   left with a bar labeled `+0.12` reads as: *"this customer pays $135.64/month, and
   specifically because of that high value, the model's predicted churn probability went up by
   0.12 (12 percentage points)."* Check the math yourself:
   `0.293 + 0.16 + 0.12 + 0.07 + 0.05 + 0.05 + 0.04 + 0.02 + 0.00 ≈ 0.799` — the base value plus
   every single bar adds up exactly to the final prediction.

**Takeaway:** think of this chart like an itemized receipt. It starts at a baseline total, then
lists exactly which features "charged" or "discounted" the final prediction, by how much, and
in what order of importance — using this specific customer's real data.


## 9. Local explanation — Force plot

The force plot shows the **exact same numbers** as the waterfall plot above — same gray
feature values, same SHAP contributions — just laid out differently, as a single horizontal
push/pull bar instead of a vertical staircase. Some people find one easier to read than the
other; it's worth learning to recognize both.

> **Colab note:** SHAP's *interactive* (JavaScript) version of this plot is often blocked by
> Colab's output sandboxing — you may see *"Visualization omitted, Javascript library not
> loaded!"* even after running `shap.initjs()`. This is a Colab restriction, not a bug in your
> code. The cell below uses the static (`matplotlib=True`) version instead, which always
> renders correctly on Colab.


In [ ]:
# Static (matplotlib) version — always renders, including on Colab
shap.force_plot(
    explainer.expected_value[1],
    shap_values_churn[idx].values,
    X_test.iloc[idx],
    matplotlib=True,
    show=False,
)
plt.tight_layout()
plt.show()

# --- Optional: interactive JS version ---
# Works in classic Jupyter Notebook, but is usually blocked by Colab's output sandboxing.
# Uncomment to try it locally:
# shap.force_plot(
#     explainer.expected_value[1],
#     shap_values_churn[idx].values,
#     X_test.iloc[idx],
# )


**📖 How to read this chart, step by step:**

1. **Find the "base value" marker.** This is the same starting point as `E[f(X)]` in the
   waterfall plot — the model's average prediction before we know anything about this specific
   customer.
2. **Follow the arrows from left to right.** Each labeled segment is one feature, with its
   gray `value = feature_name` label underneath (this customer's actual data — same meaning as
   the gray numbers in the waterfall plot).
3. **Red segments push the prediction to the right** (toward churn). The **wider** the red
   segment, the bigger that feature's SHAP contribution — same numbers as the `+0.xx` labels
   you saw in the waterfall plot, just shown as width here instead of a printed number.
4. **Blue segments push the prediction to the left** (toward staying) — width again shows the
   size of the effect.
5. **The two colors "compete," like a tug-of-war.** Red is pulling toward "churn," blue is
   pulling toward "stay." Wherever the combined arrow ends up landing — marked `f(x)` — is the
   final prediction for this customer.
6. **Cross-check with Section 8:** the same feature (e.g. `monthly_charges`) should be pushing
   in the same direction, by the same amount, in both charts. If you can match them up, you've
   understood both plot types.

**Takeaway:** use the waterfall plot when you want to see feature values and step-by-step
detail spelled out; use the force plot when you want something compact enough to embed in a
one-line dashboard summary.


## 10. Feature interaction — Dependence plot

**Question this chart answers:** *"As this ONE feature's value changes, how does its SHAP
value (its push on the prediction) change — and does that relationship also depend on ANOTHER
feature?"*

This is where SHAP goes beyond a simple importance ranking and starts to reveal *how* a
feature behaves.


In [ ]:
shap.dependence_plot(
    "satisfaction_score",
    shap_values_churn.values,
    X_test,
    show=False,
)
plt.tight_layout()
plt.show()


**📖 How to read this chart, step by step:**

1. **X-axis: the feature's actual value.** Here, `satisfaction_score`, running from low (left,
   near 0) to high (right, near 10). This is raw customer data, not a SHAP value.
2. **Y-axis: that feature's SHAP value.** Above zero = pushed that customer's prediction toward
   churn; below zero = pushed it toward staying. This axis IS the SHAP value.
3. **Each dot = one customer**, placed at (their satisfaction score, the SHAP value that score
   produced for them).
4. **Read the overall trend first, left to right.** You should see dots start **high on the
   left** (low satisfaction → positive SHAP value → pushes toward churn) and trend **downward
   to the right** (high satisfaction → negative SHAP value → pushes toward staying). This
   confirms, visually, the rule we built into the data: *lower satisfaction increases churn
   risk* — and shows the model learned a smooth, consistent version of that rule.
5. **Now look at the color.** SHAP automatically colors the dots by whichever *other* feature
   interacts most strongly with `satisfaction_score` (check the color bar's label on the right
   for which feature that is). If dots at the same x-position (same satisfaction score) have
   noticeably different y-positions depending on color, that's an **interaction effect** — e.g.
   "low satisfaction hurts a bit more when the customer also has high monthly charges."
6. **A tight, smooth line of dots** = the feature acts mostly on its own, with little
   interaction. **A scattered, fanned-out cloud at the same x-value** = the feature's effect
   really does depend on other features too.

**Takeaway:** the bar and beeswarm plots (Sections 6–7) tell you *what* matters and roughly
*which direction*. The dependence plot tells you the precise *shape* of that relationship —
including whether two features' effects are tangled together.


## 11. Bonus: model-agnostic SHAP with a non-tree model

Everything above used `TreeExplainer`, which only works for tree-based models. For **any**
other model type (logistic regression, SVM, a neural network, or even an external API), use
the general-purpose `shap.Explainer`. It automatically picks a suitable algorithm — here it
falls back to a linear/permutation-based approach for `LogisticRegression`.

> ⚠️ **Speed note:** model-agnostic explainers approximate Shapley values by resampling, which
> is much slower than `TreeExplainer`'s exact math. That's fine for a quick classroom demo on a
> small sample, but it wouldn't scale to explaining thousands of predictions per day in a real
> production system.


In [ ]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

# Use a small background sample for speed in class demos
background = shap.sample(X_train, 100, random_state=RANDOM_STATE)
explainer_lr = shap.Explainer(log_reg, background)

# Explain only a small slice of the test set for speed
shap_values_lr = explainer_lr(X_test.iloc[:50])

shap.summary_plot(shap_values_lr, X_test.iloc[:50], show=False)
plt.title("SHAP summary — Logistic Regression (model-agnostic explainer)")
plt.tight_layout()
plt.show()


**📖 How to read this chart, step by step:**

Good news — this is the **same type of beeswarm plot you already learned to read in
Section 7**. Same rules apply:

1. Rows = features, ranked by importance for this new model (Logistic Regression instead of
   Random Forest).
2. Dots = customers (just the first 50 here, to keep it fast).
3. Horizontal position = SHAP value (right = toward churn, left = toward staying).
4. Color = that customer's actual feature value (red/pink = high, blue = low).

**What's interesting to compare:** does Logistic Regression rank features similarly to the
Random Forest in Section 7, and do the color patterns point the same direction (e.g. is low
satisfaction still on the "toward churn" side)? Since we know the *true* rule we built into the
data in Section 2, we can actually judge which model recovered it more faithfully — a good
discussion point, since in the real world you rarely know the true rule in advance.


## 12. Key takeaways (for slides / discussion)

| Concept | Tool | Question it answers |
|---|---|---|
| Global importance | `shap.summary_plot(..., plot_type="bar")` | "What does the model care about overall?" |
| Global importance + direction | `shap.summary_plot(...)` (beeswarm) | "How does each feature push predictions, and for which values?" |
| Single-prediction explanation | `shap.plots.waterfall(...)` / `shap.force_plot(...)` | "Why did the model say THIS about THIS customer?" |
| Feature interactions | `shap.dependence_plot(...)` | "Does this feature behave differently depending on another feature?" |
| Any model type | `shap.Explainer(model, background)` | When you're not using a tree ensemble |

**Two number types to always keep separate (this trips up most beginners):**
- **Feature value** — the customer's actual raw data (e.g. `satisfaction_score = 2.3`). Always
  shown in gray text or on the x-axis of a dependence plot, or as dot color in a beeswarm plot.
- **SHAP value** — how much that feature value influenced the prediction (e.g. `+0.23`).
  Always the thing being measured on a bar's length/width, a dot's horizontal position, or the
  y-axis of a dependence plot.

**Discussion prompts for class:**
- Which features would you *not* want a churn model relying on, even if they're predictive
  (e.g. protected attributes)? How would SHAP help you audit for that?
- How would waterfall/force plots change how a customer retention team scripts their outreach
  calls?
- What's the tradeoff between `TreeExplainer` (exact, fast) and `KernelExplainer`/permutation
  explainers (slow, model-agnostic) in a production system explaining thousands of predictions
  per day?

### Further reading
- Lundberg & Lee, *"A Unified Approach to Interpreting Model Predictions"* (2017) — the
  original SHAP paper
- [SHAP documentation](https://shap.readthedocs.io/)
